# UCS420 — Lab Assignment 9
## Vision-Based Cognitive Systems — Google Colab Notebook

**Task 1:** Design a cognitive assistant for reading labels on medicine bottles.  
**Task 2:** Design an emotion-aware cognitive assistant using facial-expression detection.



## 0. Environment setup

In [ ]:
import os, shutil, subprocess, sys

if shutil.which("tesseract") is None:
    if shutil.which("apt-get"):
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", "tesseract-ocr"], check=True)
    else:
        raise RuntimeError("Tesseract OCR not found. Install it for your operating system first.")

subprocess.run([sys.executable, "-m", "pip", "-q", "install", "pytesseract", "opencv-python-headless", "Pillow", "matplotlib", "pandas", "scikit-image", "transformers>=4.45,<5"], check=True)
print("Setup complete. Tesseract:", shutil.which("tesseract"))

In [ ]:
from pathlib import Path
import re
import json
import numpy as np
import pandas as pd
import cv2
import pytesseract
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont, ImageEnhance
from IPython.display import display

WORKDIR = Path("UCS420_Lab9")
WORKDIR.mkdir(exist_ok=True)
print("Working directory:", WORKDIR.resolve())

---
# Task 1 — Medicine Bottle Label Reading Assistant

### Goal
The assistant performs: **input image → image enhancement → optical character recognition → key-field extraction → plain-language summary**.

For the reproducible demonstration, we generate a small **synthetic label dataset**. These labels are fictional, contain no real prescriptions, and are **not intended for medical use**. You can replace them with real photographs using the upload cell below.

### 1.1 Generate a small practice dataset of fictional labels

In [ ]:
LABEL_DIR = WORKDIR / "demo_medicine_labels"
LABEL_DIR.mkdir(exist_ok=True)

samples = [
    {"name": "DEMOCARE", "strength": "250 mg", "expiry": "12/2028", "batch": "BATCH-A", "directions": "Follow clinician instructions"},
    {"name": "SAMPLEVIT", "strength": "100 mg", "expiry": "04/2029", "batch": "BATCH-B", "directions": "See printed instructions"},
    {"name": "PRACTICIN", "strength": "500 mg", "expiry": "08/2027", "batch": "BATCH-C", "directions": "Follow clinician instructions"},
    {"name": "TESTAMIN", "strength": "50 mg",  "expiry": "11/2028", "batch": "BATCH-D", "directions": "See printed instructions"},
]

def get_font(size, bold=False):
    paths = [
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf" if bold else "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/truetype/liberation2/LiberationSans-Bold.ttf" if bold else "/usr/share/fonts/truetype/liberation2/LiberationSans-Regular.ttf",
    ]
    for path in paths:
        if Path(path).exists():
            return ImageFont.truetype(path, size)
    return ImageFont.load_default()

def make_practice_label(row, out_path):
    im = Image.new("RGB", (1040, 650), (241, 241, 237))
    d = ImageDraw.Draw(im)
    d.rounded_rectangle((55, 55, 985, 605), radius=25, fill="white", outline=(95, 105, 115), width=5)
    d.rectangle((78, 78, 961, 146), fill=(21, 64, 94))
    d.text((98, 91), "TRAINING SAMPLE - NOT FOR MEDICAL USE", font=get_font(27, True), fill="white")
    lines = [
        ("Medicine:", row["name"]),
        ("Strength:", row["strength"]),
        ("Directions:", row["directions"]),
        ("Expiry:", row["expiry"]),
        ("Batch:", row["batch"]),
        ("Warning:", "Keep out of reach of children"),
    ]
    for idx, (k, v) in enumerate(lines):
        y = 169 + idx * 65
        d.text((98, y), k, font=get_font(31, True), fill=(28, 35, 43))
        d.text((330, y), v, font=get_font(31), fill=(28, 35, 43))
    d.text((104, 576), "FICTIONAL DATASET FOR OCR DEMONSTRATION", font=get_font(17), fill=(100, 100, 100))
    im.save(out_path)
    return im

truth = []
for i, row in enumerate(samples, start=1):
    f = LABEL_DIR / f"label_{i:02d}.png"
    make_practice_label(row, f)
    truth.append({"filename": f.name, **row})

pd.DataFrame(truth).to_csv(LABEL_DIR / "ground_truth.csv", index=False)
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, row in zip(axes.ravel(), truth):
    ax.imshow(Image.open(LABEL_DIR / row["filename"]))
    ax.set_title(row["filename"])
    ax.axis("off")
plt.tight_layout(); plt.show()
print("Generated", len(truth), "fictional label images + ground_truth.csv")

### 1.2 Preprocess the photograph for OCR

The OCR engine works better when the image is readable. We enlarge the image, convert it to grayscale, denoise it, and create a black/white version. The assistant compares these versions and uses the OCR result with the best *approximate* word confidence.

In [ ]:
def prepare_label_versions(image):
    """Input: PIL image. Output: dictionary of enhanced PIL images."""
    rgb = np.array(image.convert("RGB"))
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    if gray.shape[1] < 1200:
        scale = 1200 / gray.shape[1]
        gray = cv2.resize(gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    denoised = cv2.bilateralFilter(gray, 7, 40, 40)
    thresh = cv2.adaptiveThreshold(denoised, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                   cv2.THRESH_BINARY, 41, 13)
    return {
        "grayscale": Image.fromarray(gray),
        "enhanced": Image.fromarray(denoised),
        "threshold": Image.fromarray(thresh),
    }

def get_ocr_candidate(prepared_image, config="--oem 3 --psm 6"):
    data = pytesseract.image_to_data(prepared_image, output_type=pytesseract.Output.DICT, config=config)
    words = []
    confidences = []
    for word, conf in zip(data["text"], data["conf"]):
        if word.strip():
            words.append(word)
            try:
                if float(conf) >= 0:
                    confidences.append(float(conf))
            except (ValueError, TypeError):
                pass
    full_text = pytesseract.image_to_string(prepared_image, config=config).strip()
    avg_conf = float(np.mean(confidences)) if confidences else 0.0
    return {"text": full_text, "confidence": avg_conf, "word_count": len(words)}

def read_label_text(image):
    variants = prepare_label_versions(image)
    attempts = {name: get_ocr_candidate(img) for name, img in variants.items()}
    best = max(attempts, key=lambda k: (attempts[k]["confidence"], attempts[k]["word_count"]))
    return {"variant": best, **attempts[best], "attempts": attempts, "processed": variants[best]}

example = Image.open(LABEL_DIR / truth[0]["filename"])
ocr_example = read_label_text(example)
print("Selected preprocessing:", ocr_example["variant"])
print("Mean OCR word confidence (not a medical accuracy guarantee):", round(ocr_example["confidence"], 1))
print("\nOCR text:\n", ocr_example["text"])
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].imshow(example); axes[0].set_title("Original label")
axes[1].imshow(ocr_example["processed"], cmap="gray"); axes[1].set_title("OCR-selected preprocessing")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

### 1.3 Recognize fields and generate a readable assistant response

This parser is a simple, transparent rule-based information extractor. It **only** returns values that appear in the recognized text. Text may be missed or misread, particularly on curved, reflective, or blurry real bottles.

In [ ]:
def extract_label_fields(ocr_text):
    """Extract explicitly printed fields; do not guess missing medical information."""
    lines = [re.sub(r"\s+", " ", line.strip()) for line in ocr_text.splitlines() if line.strip()]
    rules = {
        "medicine_name": r"^(?:medicine|drug|product|name)\s*[:;-]\s*(.+)$",
        "strength": r"^(?:strength|dose strength|concentration)\s*[:;-]\s*(.+)$",
        "directions": r"^(?:directions|instructions|usage|dosage)\s*[:;-]\s*(.+)$",
        "expiry": r"^(?:expiry|expiration|exp\.?|expires)\s*[:;-]\s*(.+)$",
        "batch": r"^(?:batch|lot|lot no\.?)\s*[:;-]\s*(.+)$",
        "warning": r"^(?:warning|caution)\s*[:;-]\s*(.+)$",
    }
    fields = {k: None for k in rules}
    for line in lines:
        for key, pattern in rules.items():
            match = re.match(pattern, line, flags=re.IGNORECASE)
            if match and fields[key] is None:
                fields[key] = match.group(1).strip(" .:-")
    return fields

def medicine_label_assistant(image, filename="image"):
    reading = read_label_text(image)
    fields = extract_label_fields(reading["text"])
    print("=" * 65)
    print("MEDICINE LABEL READING ASSISTANT |", filename)
    print("=" * 65)
    for k, value in fields.items():
        print(f"{k.replace('_',' ').title():15s}: {value if value else 'Not reliably detected'}")
    print(f"\nOCR word confidence (rough estimate): {reading['confidence']:.1f}/100")
    print("\nAssistant: I have transcribed the visible text and highlighted fields that I could read.")
    print("Please compare every field with the original packaging.")
    print("Do NOT use this output to identify medicine or decide dosage; ask a pharmacist if anything is unclear.")
    return {"file": filename, "fields": fields, "raw_text": reading["text"],
            "ocr_confidence": reading["confidence"], "preprocessing": reading["variant"]}

label_result = medicine_label_assistant(example, filename=truth[0]["filename"])
display(pd.DataFrame([label_result["fields"]]).T.rename(columns={0: "Detected value"}))

### 1.4 Batch test on the generated practice dataset

We check whether the recognized `medicine_name`, `strength`, `expiry`, and `batch` match the known fictional labels. These practice-set figures illustrate the pipeline; they are **not clinical validation** and will not generalize to real bottle photos.

In [ ]:
all_results = []
check_fields = ["medicine_name", "strength", "expiry", "batch"]
for item in truth:
    img = Image.open(LABEL_DIR / item["filename"])
    ocr = read_label_text(img)
    got = extract_label_fields(ocr["text"])
    row = {"filename": item["filename"], "ocr_confidence": round(ocr["confidence"], 1)}
    ground_names = {"medicine_name": "name", "strength": "strength", "expiry": "expiry", "batch": "batch"}
    for key in check_fields:
        target = str(item[ground_names[key]]).casefold().strip()
        predicted = str(got[key] or "").casefold().strip()
        row[key + "_correct"] = (target == predicted)
    all_results.append(row)

results_df = pd.DataFrame(all_results)
display(results_df)
print("Exact-match field rate on SYNTHETIC examples:",
      f"{results_df[[k + '_correct' for k in check_fields]].to_numpy().mean():.1%}")
results_df.to_csv(WORKDIR / "ocr_demo_evaluation.csv", index=False)

### 1.5 Optional: upload a real bottle-label photo

Set the switch to `True` and rerun this cell when you're ready. Take the photograph in bright light, keep the text in focus, and **cover personal details**. For Colab, the file picker opens automatically.

In [ ]:
UPLOAD_MEDICINE_IMAGE = False  # Change to True to upload your own image.

if UPLOAD_MEDICINE_IMAGE:
    try:
        from google.colab import files
        uploaded = files.upload()
        for filename, contents in uploaded.items():
            from io import BytesIO
            im = Image.open(BytesIO(contents)).convert("RGB")
            display(im)
            user_label_result = medicine_label_assistant(im, filename)
    except ImportError:
        print("Not running in Google Colab. Open an image manually with Image.open('your_image.jpg').")
else:
    print("Upload disabled. Set UPLOAD_MEDICINE_IMAGE = True to select a real photo.")

---
# Task 2 — Emotion-Aware Cognitive Assistant Using Facial Expression Detection

### Goal
**Input face image → detect face → crop face → classify the visible facial expression → provide a nonjudgmental assistant message**.

The emotion classifier is a pretrained **Vision Transformer (ViT)** from Hugging Face: [`trpakov/vit-face-expression`](https://huggingface.co/trpakov/vit-face-expression), which is trained on the **FER2013** facial-expression dataset. It predicts seven expression categories: angry, disgust, fear, happy, sad, surprise, and neutral.

**Ethics note:** A facial expression is not a reliable measurement of a person's actual emotion. Treat the output as a tentative appearance-based classification only, and use images with permission.

### 2.1 Load a sample face image and locate the face

A sample photograph from `scikit-image` is included to demonstrate the face-detection workflow. Face detection uses OpenCV's pretrained Haar cascade.

In [ ]:
from skimage import data as skdata

face_sample = Image.fromarray(skdata.astronaut()).convert("RGB")
display(face_sample)

def detect_faces(pil_image):
    rgb = np.array(pil_image.convert("RGB"))
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    cascade_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    detector = cv2.CascadeClassifier(cascade_path)
    if detector.empty():
        raise RuntimeError("OpenCV face detector could not be loaded")
    faces = detector.detectMultiScale(gray, scaleFactor=1.06, minNeighbors=5, minSize=(35, 35))
    if len(faces) == 0:
        faces = detector.detectMultiScale(gray, scaleFactor=1.03, minNeighbors=4, minSize=(30, 30))
    return sorted([tuple(map(int, f)) for f in faces], key=lambda f: f[2] * f[3], reverse=True)

def crop_largest_face(pil_image, margin=0.18):
    faces = detect_faces(pil_image)
    if not faces:
        return None, [], None
    x, y, w, h = faces[0]
    W, H = pil_image.size
    pad = int(max(w, h) * margin)
    crop = pil_image.crop((max(0, x-pad), max(0, y-pad), min(W, x+w+pad), min(H, y+h+pad)))
    return crop, faces, (x, y, w, h)

sample_crop, sample_faces, sample_box = crop_largest_face(face_sample)
print("Faces detected:", len(sample_faces))
if sample_crop is not None:
    display(sample_crop)
else:
    print("No face found. You can try a clear, front-facing portrait in the upload cell.")

### 2.2 Load the pretrained facial-expression model

The first run downloads model weights (hundreds of MB) from Hugging Face. **Internet access is required.** A GPU is optional; CPU inference is supported. No Kaggle account or manually downloaded dataset is needed.

In [ ]:
import torch
from transformers import AutoImageProcessor, AutoModelForImageClassification

MODEL_ID = "trpakov/vit-face-expression"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Inference device:", DEVICE)

processor = AutoImageProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageClassification.from_pretrained(MODEL_ID).to(DEVICE)
model.eval()
print("Expression model ready:", MODEL_ID)
print("Classes:", model.config.id2label)

### 2.3 Classify the visible expression and show a score chart

The system returns model scores. Scores are **not** guarantees about genuine feelings; if the highest score is low, the assistant treats its interpretation as uncertain.

In [ ]:
def expression_scores(face_crop):
    if face_crop is None:
        raise ValueError("A detected face crop is required")
    inputs = processor(images=face_crop, return_tensors="pt")
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.inference_mode():
        logits = model(**inputs).logits[0]
    probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()
    return sorted([{"expression": str(model.config.id2label[i]).lower(),
                    "score": float(probs[i])} for i in range(len(probs))],
                  key=lambda d: d["score"], reverse=True)

def expression_assistant_message(label, confidence):
    if confidence < 0.45:
        return "The visible expression is unclear. How would you like me to assist you?"
    messages = {
        "happy": "Your expression looks smile-like. What would you like to do next?",
        "sad": "Your expression may look downcast. I can respond at your pace—what would help?",
        "angry": "Your expression may look tense. Would you like a shorter, simpler response?",
        "fear": "Your expression may show surprise or tension. How can I help?",
        "surprise": "Your expression looks surprised. Would an explanation help?",
        "neutral": "Your expression looks neutral. What would you like to work on?",
        "disgust": "Your expression may show discomfort. Would you prefer a different approach?",
    }
    return messages.get(label, "I can help. What would you like to do next?")

def facial_expression_assistant(pil_image, image_name="image"):
    crop, faces, bbox = crop_largest_face(pil_image)
    print("=" * 66)
    print("EXPRESSION-AWARE COGNITIVE ASSISTANT |", image_name)
    print("=" * 66)
    if crop is None:
        print("No frontal face detected; expression classification was skipped.")
        print("Assistant: Please provide a clear, consented, front-facing portrait.")
        return None
    preds = expression_scores(crop)
    top = preds[0]
    print(f"Faces detected: {len(faces)} (using the largest face)")
    print(f"Visible expression class: {top['expression']} ({top['score']:.1%} model score)")
    print("Assistant:", expression_assistant_message(top["expression"], top["score"]))
    print("Reminder: appearance does not reveal someone's actual emotions.")
    # Overlay a face rectangle and show class scores.
    annotated = np.array(pil_image.convert("RGB")).copy()
    x, y, w, h = bbox
    cv2.rectangle(annotated, (x, y), (x+w, y+h), (30, 130, 230), 3)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    axes[0].imshow(annotated); axes[0].axis("off"); axes[0].set_title("Detected face")
    # Reverse so highest class appears at top.
    vals = preds[::-1]
    axes[1].barh([d["expression"] for d in vals], [d["score"] for d in vals], color="#286aa6")
    axes[1].set_xlim(0, 1); axes[1].set_xlabel("Model classification score")
    axes[1].set_title("Facial-expression scores")
    plt.tight_layout(); plt.show()
    return {"file": image_name, "top_expression": top["expression"],
            "top_score": top["score"], "faces_found": len(faces), "all_scores": preds}

if sample_crop is not None:
    sample_expression_result = facial_expression_assistant(face_sample, "skimage astronaut demo")
else:
    print("Sample face was not detected; use the optional upload cell below.")

### 2.4 Optional: upload an expression photo

Use a photo of a person who has agreed to the experiment. **Do not** interpret the result as a diagnosis or use it to make high-stakes judgments about the person.

In [ ]:
UPLOAD_FACE_IMAGE = False  # Set True for your own image in Colab.

if UPLOAD_FACE_IMAGE:
    try:
        from google.colab import files
        from io import BytesIO
        uploaded_faces = files.upload()
        for fname, contents in uploaded_faces.items():
            img = Image.open(BytesIO(contents)).convert("RGB")
            face_result = facial_expression_assistant(img, fname)
    except ImportError:
        print("Open a local image using Image.open('portrait.jpg') and call facial_expression_assistant(image).")
else:
    print("Upload disabled. Set UPLOAD_FACE_IMAGE = True to select an image.")

### 2.5 Optional evaluation on a labeled image collection

If you have consented/test facial images organized in folders by visible **expression labels** (`happy/`, `neutral/`, `sad/`, etc.), point `EXPRESSION_DATASET_DIR` to the parent folder. The code computes accuracy and a classification report **only if such a dataset exists**. It does **not** fabricate evaluation scores.

In [ ]:
EXPRESSION_DATASET_DIR = Path("expression_test_images")  # Optional folder: happy/*.jpg, sad/*.jpg, etc.
valid_exts = {".jpg", ".jpeg", ".png", ".webp"}
allowed_labels = {"happy", "sad", "neutral", "angry", "fear", "surprise", "disgust"}

if EXPRESSION_DATASET_DIR.exists():
    true_labels, predicted_labels, skipped = [], [], 0
    for class_dir in sorted(EXPRESSION_DATASET_DIR.iterdir()):
        if not class_dir.is_dir() or class_dir.name.lower() not in allowed_labels:
            continue
        for f in class_dir.rglob("*"):
            if f.suffix.lower() not in valid_exts:
                continue
            try:
                crop, _, _ = crop_largest_face(Image.open(f).convert("RGB"))
                if crop is None:
                    skipped += 1; continue
                scores = expression_scores(crop)
                true_labels.append(class_dir.name.lower())
                predicted_labels.append(scores[0]["expression"])
            except Exception as exc:
                skipped += 1
                print("Skipped", f.name, type(exc).__name__)
    if true_labels:
        from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
        print("Evaluated images:", len(true_labels), "| skipped:", skipped)
        print("Visible-expression accuracy on YOUR test set:", f"{accuracy_score(true_labels, predicted_labels):.2%}")
        print(classification_report(true_labels, predicted_labels, zero_division=0))
        ConfusionMatrixDisplay.from_predictions(true_labels, predicted_labels, xticks_rotation=45)
        plt.tight_layout(); plt.show()
    else:
        print("No valid labeled face images found; metrics not computed.")
else:
    print("No labeled expression dataset supplied. Optional evaluation skipped.")

---
# Discussion, Outcomes, and Conclusion

### Task 1: Medicine-label assistant
- **Perception:** Preprocesses images and reads printed text with OCR.
- **Knowledge extraction:** Finds only explicitly printed fields such as medicine name, strength, directions, expiry, and batch.
- **User interaction:** Returns a readable summary with uncertainty and a safety reminder.
- **Limitations:** Curved bottles, glare, small print, handwriting, image blur, and OCR errors can cause missing or incorrect fields.

### Task 2: Expression-aware assistant
- **Perception:** Detects and crops a frontal face.
- **Pattern recognition:** A trained Vision Transformer predicts a **visible expression category**.
- **Adaptive interaction:** The assistant offers a neutral next step rather than assuming what someone feels.
- **Limitations:** Lighting, pose, occlusion, demographics, and cultural expression differences can affect predictions. It is not a mind-reading system.

### Conclusion
Both examples illustrate the **perception → interpretation → response** pattern of a vision-based cognitive system. The medicine prototype extracts text from visual inputs, while the facial-expression prototype classifies visible expressions from facial crops. Both must communicate uncertainty, avoid unsupported conclusions, and respect users' safety and privacy.

### References and datasets
1. **Original assignment:** UCS420 *Lab Assignment-9 — Landing AI (Vision based Cognitive System)* (uploaded PDF).
2. Tesseract OCR / `pytesseract`: https://github.com/madmaze/pytesseract
3. OpenCV Haar face detection: https://docs.opencv.org/4.x/db/d28/tutorial_cascade_classifier.html
4. Facial expression model: https://huggingface.co/trpakov/vit-face-expression (ViT trained on FER2013).
5. FER2013 dataset page: https://www.kaggle.com/datasets/msambare/fer2013
6. Sample face image: https://scikit-image.org/docs/stable/api/skimage.data.html#skimage.data.astronaut

**Reproducibility:** The medicine-label images are generated **synthetically in this notebook**. The emotion model's FER2013 training is provided by its model author; this notebook does not retrain that model. No performance claim is made about real medicine labels or inferred emotions.